# Phase 3 - Hyperparameter Optimization

In [ ]:
# Public Imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns
import sys
from sklearn.model_selection import train_test_split


# Custom Imports
src_dir = Path("../src")
sys.path.insert(0, str(src_dir))

from evaluate import *
from model import *
# from train import *

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


# Notebook configuration



## Load Training and Test Data

### Via np export (Comment out if not used)

In [9]:
# Define paths for data and models
data_dir = Path("../data/processed")

X_train_path = data_dir / "X_train.npy"
X_test_path = data_dir / "X_test.npy"
y_train_path = data_dir / "y_train.npy"
y_test_path = data_dir / "y_test.npy"

# Load the data
X_train = np.load(X_train_path)
X_test = np.load(X_test_path)
y_train = np.load(y_train_path)
y_test = np.load(y_test_path)

# Display the first 5 rows of the training and testing sets
# print("Train X Set:")
# display(X_train[:1])

# print("Train y Set:")
# display(y_train[:5])

# print("Test X Set:")
# display(X_test[:1])

# print("Test y Set:")
# display(y_test[:5])


# Print the shapes and data types of the training and testing sets
print("Data Summary:")
print("X_train shape:", X_train.shape)
print("X_test shapen:", X_test.shape)
print("\nX_train dtype:", X_train.dtype)
print("X_test dtype:", X_test.dtype)

print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining labels:")
print(np.unique(y_train, return_counts=True))
print("Testing labels:")
print(np.unique(y_test, return_counts=True))


# Split the training data into a fitting set and a validation set 
print("\nSplitting training data into fitting and validation sets...")
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Training set for fitting:   {X_train.shape}")
print(f"Validation set for tuning: {X_val.shape}")
print(f"Test set (held-out):       {X_test.shape}")

Data Summary:
X_train shape: (61482, 118)
X_test shapen: (84104, 118)

X_train dtype: float32
X_test dtype: float32

y_train shape: (61482,)
y_test shape: (84104,)

Training labels:
(array([0]), array([61482]))
Testing labels:
(array([0, 1]), array([26350, 57754]))

Splitting training data into fitting and validation sets...
Training set for fitting:   (61482, 118)
Validation set for tuning: (12297, 118)
Test set (held-out):       (84104, 118)


## Hyperparameter Grid Setup (3.1)

In [1]:
import itertools
import random

# --------------------------------------------------------------------------
# Data usage policy for this notebook
#
# Use the same loaded arrays as baseline notebook:
# - train on full X_train / y_train
# - use X_test / y_test as validation during hyperparameter tuning
# --------------------------------------------------------------------------
X_train_fit = X_train
X_val = X_test
y_train_fit = y_train
y_val = y_test


# --------------------------------------------------------------------------
# Hyperparameter search space
#
# latent_dim     - size of the autoencoder bottleneck (the key architecture
#                  knob for anomaly detection: too large -> reconstructs
#                  anomalies too well; too small -> can't reconstruct normal
#                  data well either)
# learning_rate  - Adam step size
# batch_size     - training batch size
# dropout_rate   - dropout after each hidden layer (0.0 = disabled)
# n_hidden_layers- number of hidden layers on EACH side of the bottleneck
#                  (encoder and decoder are mirrored), i.e. "depth"
# --------------------------------------------------------------------------
search_space = {
    'latent_dim': [8, 16, 32, 64],
    'learning_rate': [1e-2, 1e-3, 1e-4],
    'batch_size': [32, 64, 128],
    'dropout_rate': [0.0, 0.1, 0.2, 0.3],
    'n_hidden_layers': [1, 2, 3],
}

n_total_combos = 1
for v in search_space.values():
    n_total_combos *= len(v)
print(f"Full grid size: {n_total_combos} combinations")


def sample_random_configs(space, n_samples, seed=42):
    """
    Randomly sample n_samples unique hyperparameter configurations from the
    search space. We use random search instead of exhaustive grid search:
    with a large number of combinations, a full grid is not tractable to train in a
    course-project timeframe, and random search is known to find
    comparably good configurations with far fewer trials.
    """
    rng = random.Random(seed)
    keys = list(space.keys())
    all_combos = list(itertools.product(*space.values()))
    rng.shuffle(all_combos)
    sampled = all_combos[:n_samples]
    return [dict(zip(keys, combo)) for combo in sampled]


N_CONFIGS = 100  # search budget; increase if you have compute/time to spare
configs = sample_random_configs(search_space, N_CONFIGS)
print(f"Sampled {len(configs)} of {n_total_combos} possible configurations")

Full grid size: 432 combinations
Sampled 100 of 432 possible configurations


In [33]:
# Test code cells here
print("Example sampled configs:")
for cfg in configs[:5]:
    print(cfg)

Example sampled configs:
{'latent_dim': 64, 'learning_rate': 0.0001, 'batch_size': 64, 'dropout_rate': 0.3, 'n_hidden_layers': 1}
{'latent_dim': 32, 'learning_rate': 0.01, 'batch_size': 64, 'dropout_rate': 0.0, 'n_hidden_layers': 1}
{'latent_dim': 32, 'learning_rate': 0.001, 'batch_size': 128, 'dropout_rate': 0.0, 'n_hidden_layers': 3}
{'latent_dim': 8, 'learning_rate': 0.0001, 'batch_size': 64, 'dropout_rate': 0.2, 'n_hidden_layers': 3}
{'latent_dim': 16, 'learning_rate': 0.01, 'batch_size': 128, 'dropout_rate': 0.2, 'n_hidden_layers': 2}


## Model Tuning & Architecture Testing (3.2)

In [ ]:
# Develop functions here (Finsihed function go into the .py files in the src folder)

from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import time

INPUT_DIM = X_train.shape[1]


def build_autoencoder(input_dim, latent_dim, dropout_rate=0.0, n_hidden_layers=1, activation='relu'):
    """
    Build a symmetric (undercomplete) autoencoder.

    Hidden layer widths shrink geometrically from input_dim down to
    latent_dim across n_hidden_layers steps, then mirror back up for the
    decoder. This lets 'n_hidden_layers' act as a single "depth" knob in
    the search space instead of hardcoding fixed layer sizes.
    """
    if n_hidden_layers > 0:
         widths = np.linspace(np.log(input_dim), np.log(max(latent_dim, 2)), n_hidden_layers + 2)
                # cast to native Python int: numpy.int64 fails Keras's `units` validation
                # ("expected a positive integer") even though the value itself is fine
         widths = [int(w) for w in np.exp(widths).astype(int)[1:-1]]
    else:
        widths = []

    model = Sequential(  
        name=f"autoencoder_ld{latent_dim}_L{n_hidden_layers}_do{dropout_rate}"
    )
    model.add(layers.Input(shape=(input_dim,)))

    # Encoder
    for w in widths:
        model.add(layers.Dense(w, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(latent_dim, activation=activation, name="bottleneck"))

    # Decoder (mirror of encoder)
    for w in reversed(widths):
        model.add(layers.Dense(w, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(input_dim, activation='linear'))  # linear output for reconstruction

    return model


def train_and_score_config(config, X_fit, X_val, epochs=50, patience=5, verbose=0):
    """
    Train one autoencoder configuration (reconstruction task: X -> X) and
    return the trained model, its training history, and its best
    validation reconstruction loss (MSE).
    """
    model = build_autoencoder(
        input_dim=INPUT_DIM,
        latent_dim=config['latent_dim'],
        dropout_rate=config['dropout_rate'],
        n_hidden_layers=config['n_hidden_layers'],
    )
    model.compile(optimizer=Adam(learning_rate=config['learning_rate']), loss='mse')

    early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    history = model.fit(
        X_fit, X_fit,
        validation_data=(X_val, X_val),
        epochs=epochs,
        batch_size=config['batch_size'],
        callbacks=[early_stop],
        verbose=verbose,
    )

    best_val_loss = float(min(history.history['val_loss']))
    return model, history, best_val_loss

In [35]:
# Test code cells here
X_fit_ae = X_train
X_val_ae = X_test

print(f"Training autoencoder on full training set: {X_fit_ae.shape[0]} samples")
print(f"Validation during tuning uses test set:    {X_val_ae.shape[0]} samples")

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

results = []
best_val_loss = np.inf
best_model = None
best_config = None

for i, config in enumerate(configs):
    start = time.time()
    print(f"\n[{i+1}/{len(configs)}] Training config: {config}")

    try:
        model, history, val_loss = train_and_score_config(
            config, X_fit_ae, X_val_ae, epochs=50, patience=5
        )
    except Exception as e:
        print(f"  Failed: {e}")
        continue

    elapsed = time.time() - start
    print(f"  val_loss={val_loss:.6f}  ({elapsed:.1f}s, {len(history.history['loss'])} epochs run)")

    results.append({**config, 'val_loss': val_loss, 'train_time_s': elapsed})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model
        best_config = config
        best_model.save(models_dir / "autoencoder_model_best.keras")
        print(f"  --> New best model saved (val_loss={val_loss:.6f})")

if not results:
    print("\nAll configurations failed to train -- check the error messages above "
          "before proceeding (results_df would be empty).")
else:
    results_df = pd.DataFrame(results).sort_values('val_loss').reset_index(drop=True)
    results_df.to_csv(models_dir / "hyperparam_search_results.csv", index=False)

    print("\nTop 10 configurations by validation loss:")
    display(results_df.head(10))
    print(f"\nBest config: {best_config}")
    print(f"Best validation loss: {best_val_loss:.6f}")

Training autoencoder on full training set: 61482 samples
Validation during tuning uses test set:    84104 samples

[1/100] Training config: {'latent_dim': 64, 'learning_rate': 0.0001, 'batch_size': 64, 'dropout_rate': 0.3, 'n_hidden_layers': 1}
  val_loss=3.430180  (34.7s, 42 epochs run)
  --> New best model saved (val_loss=3.430180)

[2/100] Training config: {'latent_dim': 32, 'learning_rate': 0.01, 'batch_size': 64, 'dropout_rate': 0.0, 'n_hidden_layers': 1}
  val_loss=10.767039  (6.0s, 8 epochs run)

[3/100] Training config: {'latent_dim': 32, 'learning_rate': 0.001, 'batch_size': 128, 'dropout_rate': 0.0, 'n_hidden_layers': 3}
  val_loss=2.849854  (9.0s, 14 epochs run)
  --> New best model saved (val_loss=2.849854)

[4/100] Training config: {'latent_dim': 8, 'learning_rate': 0.0001, 'batch_size': 64, 'dropout_rate': 0.2, 'n_hidden_layers': 3}
  val_loss=66.819290  (7.1s, 7 epochs run)

[5/100] Training config: {'latent_dim': 16, 'learning_rate': 0.01, 'batch_size': 128, 'dropout_ra

,latent_dim,learning_rate,batch_size,dropout_rate,n_hidden_layers,val_loss,train_time_s
0,32,0.0001,32,0.0,1,0.465900,60.520106
1,64,0.0001,128,0.0,2,0.783180,30.130646
2,32,0.0001,128,0.0,1,0.875210,21.107701
3,32,0.0001,32,0.0,2,1.286723,34.075064
4,16,0.0010,32,0.0,1,1.636561,19.991288
5,32,0.0001,32,0.1,1,1.639883,465.258742
6,16,0.0010,64,0.0,1,1.657991,16.587461
7,16,0.0010,64,0.0,3,1.748578,16.608123
8,64,0.0010,128,0.0,2,1.759040,9.757756
9,16,0.0001,64,0.0,1,1.790401,31.764969



Best config: {'latent_dim': 32, 'learning_rate': 0.0001, 'batch_size': 32, 'dropout_rate': 0.0, 'n_hidden_layers': 1}
Best validation loss: 0.465900


In [ ]:
if 'results_df' not in dir() or results_df.empty:
    print("results_df is empty -- run the search cell above first.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1. Latent dimension vs val_loss, colored by depth, sized by train time
    sns.scatterplot(
        data=results_df, x='latent_dim', y='val_loss', hue='n_hidden_layers',
        size='train_time_s', palette='viridis', ax=axes[0, 0], legend='brief'
    )
    axes[0, 0].set_title('Validation Loss vs Latent Dimension')
    axes[0, 0].set_yscale('log')

    # 2. Learning rate vs val_loss
    sns.boxplot(data=results_df, x='learning_rate', y='val_loss', ax=axes[0, 1])
    axes[0, 1].set_title('Validation Loss by Learning Rate')
    axes[0, 1].set_yscale('log')

    # 3. Batch size vs val_loss
    sns.boxplot(data=results_df, x='batch_size', y='val_loss', ax=axes[0, 2])
    axes[0, 2].set_title('Validation Loss by Batch Size')
    axes[0, 2].set_yscale('log')

    # 4. Dropout rate vs val_loss
    sns.boxplot(data=results_df, x='dropout_rate', y='val_loss', ax=axes[1, 0])
    axes[1, 0].set_title('Validation Loss by Dropout Rate')
    axes[1, 0].set_yscale('log')

    # 5. Network depth vs val_loss
    sns.boxplot(data=results_df, x='n_hidden_layers', y='val_loss', ax=axes[1, 1])
    axes[1, 1].set_title('Validation Loss by Network Depth')
    axes[1, 1].set_yscale('log')

    # 6. Top-10 configurations, ranked
    top_n = results_df.head(10).copy()
    top_n['config_label'] = top_n.apply(
        lambda r: f"ld={int(r.latent_dim)},lr={r.learning_rate},bs={int(r.batch_size)},"
                  f"do={r.dropout_rate},L={int(r.n_hidden_layers)}", axis=1
    )
    sns.barplot(data=top_n, y='config_label', x='val_loss', ax=axes[1, 2], palette='crest')
    axes[1, 2].set_title('Top 10 Configurations')
    axes[1, 2].set_xlabel('Validation Loss')
    axes[1, 2].set_ylabel('')

    plt.tight_layout()
    plt.show()

    # Correlation of each hyperparameter with validation loss
    corr_cols = ['latent_dim', 'learning_rate', 'batch_size', 'dropout_rate', 'n_hidden_layers', 'val_loss']
    corr = results_df[corr_cols].corr()

    plt.figure(figsize=(7, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title('Hyperparameter Correlation Matrix')
    plt.tight_layout()
    plt.show()

    val_loss_corr = corr['val_loss'].drop('val_loss').sort_values(key=abs, ascending=False)
    print("Correlation of each hyperparameter with validation loss (sorted by strength):")
    print(val_loss_corr)

results_df is empty -- run the search cell above first.


## Evalute Model Performance

In [ ]:
print("Generating anomaly metrics and threshold")
# threshold, y_pred = generate_anomaly_metrics_and_threshold(model, X_train, X_test, y_test, percentile=THRESHOLD_PERCENTILE)

Generating anomaly metrics and threshold
